# Python程序设计（项目实战） · 工单P4.1《多源数据采集系统》 — 教师参考版
第6周 | 参考实现可直接运行验证 | 答案请勿下发学生

In [1]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())


工作目录: /Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/03-Python程序设计-项目实战


## 最小三线程采集演示（完整版见 starter W06_collector.py）

In [2]:

# --- P4.1 最小三线程演示: 采集→Queue→处理→统计(运行约10秒) ---
import subprocess, sys, time, atexit, queue, threading
ps = [subprocess.Popen([sys.executable,
        "../02-程序设计基础/虚拟产线/mini_modbus_server.py", str(p), str(u)])
      for p, u in [(5020, 1), (5021, 2), (5022, 3)]]
atexit.register(lambda: [p.terminate() for p in ps])
time.sleep(1.2)
from pymodbus.client import ModbusTcpClient

q = queue.Queue(maxsize=1000)
stat = {"ok": 0, "fail": 0, "got": 0}
stop = threading.Event()

def collect():
    clients = [ModbusTcpClient(host="127.0.0.1", port=p) for p in (5020, 5021, 5022)]
    [c.connect() for c in clients]
    while not stop.is_set():
        for c in clients:
            try:
                rr = c.read_holding_registers(address=0, count=5)
                q.put(rr.registers, timeout=1); stat["ok"] += 1
            except Exception:
                stat["fail"] += 1
        time.sleep(0.5)

def process():
    while not stop.is_set() or not q.empty():
        try:
            rows = q.get(timeout=1); stat["got"] += 1
        except queue.Empty:
            pass

t1, t2 = threading.Thread(target=collect), threading.Thread(target=process)
t1.start(); t2.start(); time.sleep(8); stop.set(); t1.join(); t2.join()
total = stat["ok"] + stat["fail"]
print(f"采集率: {stat['ok']}/{total} = {stat['ok']/max(total,1):.1%} | 入队处理 {stat['got']} 条")
[p.terminate() for p in ps]
print("演示完成; 完整三线程系统(断线重连/入库)见 学生代码框架/W06_collector.py")


采集率: 48/48 = 100.0% | 入队处理 48 条
演示完成; 完整三线程系统(断线重连/入库)见 学生代码框架/W06_collector.py


## 学生版TODO → 教师答案对照
完整三线程参考实现见 starter `W06_collector.py`（Modbus轮询/MQTT订阅/SQLite入库三线程 +
Queue汇流 + 去重与4.5mm/s告警）。学生版TODO对应：

| TODO | 题面（学生看到的指导） |
|------|----------------------|
| `?` | （在此处编写你的代码——依据上方注释/文档字符串） |
| `?` | （在此处编写你的代码——依据上方注释/文档字符串） |
| `?` | （在此处编写你的代码——依据上方注释/文档字符串） |

> 发放纪律：本对照表**勿随学生版下发**；课堂按层放提示。

## 完整参考实现（starter即答案）

In [3]:
# -*- coding: utf-8 -*-
"""P4.1 多源工业数据采集系统 —— 三线程架构骨架(已给, 聚焦协议集成与可靠性)
前置: python ../工具/plc_simulator.py  (三台PLC: 5020/5021/5022)
     python ../02-Python编程基础/虚拟产线/mini_mqtt_broker.py
"""
import json
import logging
import queue
import sqlite3
import threading
import time

from pymodbus.client import ModbusTcpClient as Client
import paho.mqtt.client as mqtt

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s %(threadName)s %(levelname)s %(message)s")
log = logging.getLogger("collector")

PLCS = [("127.0.0.1", 5020), ("127.0.0.1", 5021), ("127.0.0.1", 5022)]
NAMES = ["temp_c", "press_mpa", "vib_mm_s", "rpm", "status"]


class ModbusCollector(threading.Thread):
    """TODO-1: 轮询三台PLC(0.5s), 读5寄存器÷10还原, 放入Queue
    可靠性: 连接失败退避重试(1s→2s→4s封顶); 2号PLC会随机掉线(实测点)"""

    def __init__(self, q: queue.Queue):
        super().__init__(daemon=True, name="Modbus")
        self.q, self.stop = q, threading.Event()
        self.stat = {"ok": 0, "fail": 0}

    def read_one(self, host, port):
        # === 你的代码: Client(host,port) → read_holding_registers(0,5) ===
        # 返回 dict(含source=f"plc@{port}") 或 None
        raise NotImplementedError

    def run(self):
        backoff = 1
        while not self.stop.is_set():
            got_any = False
            for host, port in PLCS:
                data = self.read_one(host, port)
                if data:
                    self.stat["ok"] += 1
                    self.q.put(data, timeout=1)
                    got_any = True
                else:
                    self.stat["fail"] += 1
            backoff = 1 if got_any else min(backoff * 2, 4)
            time.sleep(0.5 if got_any else backoff)


class MQTTSubscriber(threading.Thread):
    """TODO-2: 订阅 device/+/status, 消息JSON解析后放入Queue"""

    def __init__(self, q):
        super().__init__(daemon=True, name="MQTT")
        self.q = q

    def run(self):
        c = mqtt.Client()

        def on_message(_c, _u, msg):
            # === 你的代码: json.loads → dict加source="mqtt" → q.put ===
            pass

        c.on_message = on_message
        c.connect("127.0.0.1", 1883)
        c.subscribe("device/+/status", qos=0)
        c.loop_forever()


class DataProcessor(threading.Thread):
    """TODO-3: 消费Queue → 去重(同source+ts跳过) → 振动>4.5告警 → 批量入库SQLite"""

    def __init__(self, q, batch=50):
        super().__init__(daemon=True, name="Proc")
        self.q, self.batch, self.buf = q, batch, []

    def run(self):
        db = sqlite3.connect("buffer.db", check_same_thread=False)
        db.execute("CREATE TABLE IF NOT EXISTS records("
                   "source TEXT, ts REAL, temp REAL, vib REAL)")
        seen = set()
        while True:
            try:
                d = self.q.get(timeout=5)
            except queue.Empty:
                self.flush(db)
                continue
            key = (d.get("source"), round(d.get("ts", 0), 1))
            # === 你的代码: 去重/告警/入buf; len(buf)>=batch时flush ===
            self.flush(db)

    def flush(self, db):
        if self.buf:
            db.executemany("INSERT INTO records VALUES(?,?,?,?)", self.buf)
            db.commit()
            self.buf.clear()


def main():
    q = queue.Queue(maxsize=10000)
    threads = [ModbusCollector(q), MQTTSubscriber(q), DataProcessor(q)]
    for t in threads:
        t.start()
    print("采集30秒后输出统计(Ctrl+C提前结束)...")
    try:
        time.sleep(30)
    except KeyboardInterrupt:
        pass
    threads[0].stop.set()
    s = threads[0].stat
    total = s["ok"] + s["fail"]
    print(f"采集率: {s['ok']}/{total} = {s['ok'] / max(total, 1):.1%} (目标≥95%)")


if __name__ == "__main__":
    main()


Exception in thread /var/folders/6j/khvhkzp55m30fx539sm9tggr0000gn/T/ipykernel_66788/3960020729.py:62: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  c = mqtt.Client()
Modbus:
Exception in thread MQTT:
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.14/3.14.6/Frameworks/Python.framework/Versions/3.14/lib/python3.14/threading.py", line 1082, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "/var/folders/6j/khvhkzp55m30fx539sm9tggr0000gn/T/ipykernel_66788/3960020729.py", line 43, in run
    data = self.read_one(host, port)
  File "/var/folders/6j/khvhkzp55m30fx539sm9tggr0000gn/T/ipykernel_66788/3960020729.py", line 36, in read_one
    raise NotImplementedError
NotImplementedError


Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.14/3.14.6/Frameworks/Python.framework/Versions/3.14/lib/python3.14/threading.py", line 1082, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "/var/folders/6j/khvhkzp55m30fx539sm9tggr0000gn/T/ipykernel_66788/3960020729.py", line 69, in run
    c.connect("127.0.0.1", 1883)
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/jupyterlab/4.5.7_2/libexec/lib/python3.14/site-packages/paho/mqtt/client.py", line 1435, in connect
    return self.reconnect()
           ~~~~~~~~~~~~~~^^
  File "/opt/homebrew/Cellar/jupyterlab/4.5.7_2/libexec/lib/python3.14/site-packages/paho/mqtt/client.py", line 1598, in reconnect
    self._sock = self._create_socket()
                 ~~~~~~~~~~~~~~~~~~~^^
  File "/opt/homebrew/Cellar/jupyterlab/4.5.7_2/libexec/lib/python3.14/site-packages/paho/mqtt/client.py", line 4609, in _create_socket
    sock = self._create_socket_connection()
  Fil

采集30秒后输出统计(Ctrl+C提前结束)...


采集率: 0/0 = 0.0% (目标≥95%)


## 多解对比（逻辑同构烟测，真连PLC跑starter W06_collector.py）
| 解 | 路线 | 结论 |
|----|------|------|
| A | 三线程+Queue | 阻塞IO直观，教材主线（payload同构已验） |
| B | asyncio异步协程 | IO密集高并发扩展路径 |

> 决策素材：三台PLC→A；上百测点→B。